##
#**Week 1**

In [ ]:
!pip install pandas spacy sentence-transformers faiss-cpu requests
!python -m spacy download en_core_web_sm

(pip install output omitted for brevity)

In [ ]:
# --- Imports ---
import requests
import pandas as pd
import time
import json
from datetime import datetime, timezone

# --- Function to fetch posts, with retry protection ---
def fetch_posts(subreddit, after, before, limit=100, max_retries=5):
    url = "https://arctic-shift.photon-reddit.com/api/posts/search"
    all_posts = []
    params = {"subreddit": subreddit, "after": after, "before": before, "limit": limit, "sort": "asc"}

    while True:
        retries = 0
        response = None
        while retries < max_retries:
            try:
                response = requests.get(url, params=params, timeout=30)
                if response.status_code == 200:
                    break
                print(f"Status {response.status_code}, retrying... ({retries+1}/{max_retries})")
            except requests.exceptions.RequestException as e:
                print(f"Connection error, retrying... ({e})")
                response = None
            time.sleep(5)
            retries += 1

        if response is None or response.status_code != 200:
            print("Stopping, saving what we have.")
            break

        data = response.json()
        posts = data.get("data", [])
        if not posts:
            print("No more posts — done.")
            break

        all_posts.extend(posts)
        print(f"Posts collected: {len(all_posts)}")

        with open("posts_progress.json", "w") as f:
            json.dump(all_posts, f)

        last_created = posts[-1]["created_utc"]
        next_date = datetime.fromtimestamp(last_created, tz=timezone.utc)
        params["after"] = next_date.strftime("%Y-%m-%dT%H:%M:%S")
        time.sleep(2.5)

        if len(posts) < limit:
            break

    return all_posts

# --- Run it ---
raw_posts = fetch_posts("artificial", "2026-06-01", "2026-06-30")
posts_df = pd.DataFrame(raw_posts)

print(f"\nTOTAL POSTS: {len(raw_posts)}")
print("Posts shape:", posts_df.shape)

Posts collected: 100
...
Posts collected: 2317

TOTAL POSTS: 2317
Posts shape: (2317, 117)

In [ ]:
# --- Function to fetch comments, with retry protection ---
def fetch_comments(subreddit, after, before, limit=100, max_retries=5):
    url = "https://arctic-shift.photon-reddit.com/api/comments/search"
    all_comments = []
    params = {"subreddit": subreddit, "after": after, "before": before, "limit": limit, "sort": "asc"}

    while True:
        retries = 0
        response = None
        while retries < max_retries:
            try:
                response = requests.get(url, params=params, timeout=30)
                if response.status_code == 200:
                    break
                print(f"Status {response.status_code}, retrying... ({retries+1}/{max_retries})")
            except requests.exceptions.RequestException as e:
                print(f"Connection error, retrying... ({e})")
                response = None
            time.sleep(5)
            retries += 1

        if response is None or response.status_code != 200:
            print("Stopping, saving what we have.")
            break

        data = response.json()
        comments = data.get("data", [])
        if not comments:
            print("No more comments — done.")
            break

        all_comments.extend(comments)
        print(f"Comments collected: {len(all_comments)}")

        with open("comments_progress.json", "w") as f:
            json.dump(all_comments, f)

        last_created = comments[-1]["created_utc"]
        next_date = datetime.fromtimestamp(last_created, tz=timezone.utc)
        params["after"] = next_date.strftime("%Y-%m-%dT%H:%M:%S")
        time.sleep(2.5)

        if len(comments) < limit:
            break

    return all_comments

# --- Run it ---
raw_comments = fetch_comments("artificial", "2026-06-01", "2026-06-30")
comments_df = pd.DataFrame(raw_comments)

print(f"\nTOTAL COMMENTS: {len(raw_comments)}")
print("Comments shape:", comments_df.shape)

Comments collected: 100
...
Comments collected: 20447

TOTAL COMMENTS: 20447
Comments shape: (20447, 75)

In [ ]:
import re

# --- 1. Combine post title + body into one "text" column ---
posts_df["text"] = posts_df["title"].fillna("") + " " + posts_df["selftext"].fillna("")
comments_df["text"] = comments_df["body"].fillna("")

# --- 2. Remove deleted/removed content ---
deleted_markers = ["[deleted]", "[removed]"]

posts_clean = posts_df[~posts_df["text"].str.strip().isin(deleted_markers)].copy()
comments_clean = comments_df[~comments_df["text"].str.strip().isin(deleted_markers)].copy()

print(f"Posts after removing deleted/removed: {len(posts_clean)} (was {len(posts_df)})")
print(f"Comments after removing deleted/removed: {len(comments_clean)} (was {len(comments_df)})")

# --- 3. Remove bot comments ---
bot_authors = ["AutoModerator"]
comments_clean = comments_clean[~comments_clean["author"].isin(bot_authors)].copy()
print(f"Comments after removing bots: {len(comments_clean)}")

# --- 4. Strip URLs and markdown formatting using regex ---
def clean_text(text):
    text = str(text)
    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = re.sub(r"\*\*(.*?)\*\*", r"\1", text)
    text = re.sub(r"\*(.*?)\*", r"\1", text)
    text = re.sub(r"\[(.*?)\]\(.*?\)", r"\1", text)
    text = re.sub(r"[#>`~]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

posts_clean["clean_text"] = posts_clean["text"].apply(clean_text)
comments_clean["clean_text"] = comments_clean["text"].apply(clean_text)

# --- 5. Remove rows that are empty after cleaning ---
posts_clean = posts_clean[posts_clean["clean_text"].str.len() > 0]
comments_clean = comments_clean[comments_clean["clean_text"].str.len() > 0]

# --- 6. Deduplicate ---
posts_clean = posts_clean.drop_duplicates(subset="clean_text")
comments_clean = comments_clean.drop_duplicates(subset="clean_text")

print(f"\nFinal posts: {len(posts_clean)}")
print(f"Final comments: {len(comments_clean)}")

# --- 7. Preview ---
posts_clean[["clean_text"]].head()

Posts after removing deleted/removed: 2317 (was 2317)
Comments after removing deleted/removed: 18791 (was 20447)
Comments after removing bots: 18791

Final posts: 2264
Final comments: 18397

In [ ]:
import spacy
from collections import Counter

nlp = spacy.load("en_core_web_sm")

def tokenize(text):
    doc = nlp(text.lower())
    tokens = [token.text for token in doc if token.is_alpha and not token.is_stop]
    return tokens

sample_comments = comments_clean["clean_text"].sample(min(2000, len(comments_clean)), random_state=42)

all_tokens = []
for text in sample_comments:
    all_tokens.extend(tokenize(text))

print(f"Total tokens (from {len(sample_comments)} sampled comments): {len(all_tokens)}")

word_freq = Counter(all_tokens)
top_terms = word_freq.most_common(20)

print("\nTop 20 most common words:")
for word, count in top_terms:
    print(f"{word}: {count}")

Total tokens (from 2000 sampled comments): 43125

Top 20 most common words:
ai: 925
like: 380
people: 344
use: 282
think: 254
model: 230
time: 203
work: 192
actually: 190
way: 180
good: 172
real: 160
data: 160
need: 158
things: 150
models: 150
thing: 145
human: 140
know: 136
right: 125

##
#**Week 2**

In [ ]:
!pip install pandas sentence-transformers chromadb

(pip install output omitted for brevity)

In [ ]:
print("Posts:", posts_clean.shape)
print("Comments:", comments_clean.shape)

Posts: (2264, 119)
Comments: (18397, 77)

In [ ]:
import re

def chunk_text(text, max_chunk_size=200, overlap=20):
    """
    Recursively splits text into chunks of roughly max_chunk_size characters.
    Tries paragraph breaks first, then sentences, then words.
    """
    if len(text) <= max_chunk_size:
        return [text] if text.strip() else []

    paragraphs = text.split("\n\n")
    if len(paragraphs) > 1:
        chunks = []
        for para in paragraphs:
            chunks.extend(chunk_text(para, max_chunk_size, overlap))
        return chunks

    sentences = re.split(r'(?<=[.!?])\s+', text)
    if len(sentences) > 1:
        chunks = []
        current = ""
        for sentence in sentences:
            if len(current) + len(sentence) <= max_chunk_size:
                current += (" " if current else "") + sentence
            else:
                if current:
                    chunks.append(current)
                current = current[-overlap:] + " " + sentence if current else sentence
        if current:
            chunks.append(current)
        return chunks

    words = text.split()
    chunks = []
    current = ""
    for word in words:
        if len(current) + len(word) + 1 <= max_chunk_size:
            current += (" " if current else "") + word
        else:
            chunks.append(current)
            current = word
    if current:
        chunks.append(current)
    return chunks


# ---- Unit tests ----
def test_chunk_text():
    result = chunk_text("Hello world.", max_chunk_size=200)
    assert result == ["Hello world."], f"Test 1 failed: {result}"

    result = chunk_text("", max_chunk_size=200)
    assert result == [], f"Test 2 failed: {result}"

    long_text = "This is a sentence. " * 30
    result = chunk_text(long_text, max_chunk_size=100)
    assert len(result) > 1, f"Test 3 failed: got {len(result)} chunk(s)"

    for chunk in result:
        assert len(chunk) <= 150, f"Test 4 failed: chunk too long ({len(chunk)} chars): {chunk}"

    print("All unit tests passed!")

test_chunk_text()

# ---- Apply chunking to real comments ----
all_chunks = []
chunk_sources = []

for idx, row in comments_clean.iterrows():
    text = row["clean_text"]

    if pd.isna(text):
        continue

    text = str(text)

    chunks = chunk_text(text, max_chunk_size=300, overlap=30)
    for chunk in chunks:
        all_chunks.append(chunk)
        chunk_sources.append(row["id"] if "id" in comments_clean.columns else idx)

print(f"\nTotal comments: {len(comments_clean)}")
print(f"Total chunks generated: {len(all_chunks)}")
print(f"\nSample chunk: {all_chunks[0]}")

All unit tests passed!

Total comments: 18397
Total chunks generated: 30508

Sample chunk: That person will just make the model worse than it started

In [ ]:
from sentence_transformers import SentenceTransformer
import time

model = SentenceTransformer("all-MiniLM-L6-v2")

sample_size = 5000
chunks_sample = all_chunks[:sample_size]

print(f"Embedding {len(chunks_sample)} chunks...")

start_time = time.time()
embeddings = model.encode(chunks_sample, show_progress_bar=True, batch_size=64)
end_time = time.time()

elapsed = end_time - start_time
print(f"\nDone.")
print(f"Total time: {elapsed:.2f} seconds")
print(f"Chunks per second: {len(chunks_sample) / elapsed:.2f}")
print(f"Embedding shape: {embeddings.shape}")

Embedding 5000 chunks...

Done.
Total time: 3.18 seconds
Chunks per second: 1574.26
Embedding shape: (5000, 384)

In [ ]:
print(f"Embedding all {len(all_chunks)} chunks...")

start_time = time.time()
embeddings = model.encode(all_chunks, show_progress_bar=True, batch_size=64)
end_time = time.time()

elapsed = end_time - start_time
print(f"\nDone.")
print(f"Total time: {elapsed:.2f} seconds")
print(f"Chunks per second: {len(all_chunks) / elapsed:.2f}")
print(f"Embedding shape: {embeddings.shape}")

Embedding all 30508 chunks...

Done.
Total time: 21.10 seconds
Chunks per second: 1445.94
Embedding shape: (30508, 384)

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model = SentenceTransformer("all-MiniLM-L6-v2", device=device)
model.half()

start_time = time.time()
embeddings = model.encode(
    all_chunks,
    show_progress_bar=True,
    batch_size=256,
    convert_to_numpy=True
)
end_time = time.time()

elapsed = end_time - start_time
print(f"\nDone.")
print(f"Total time: {elapsed:.2f} seconds")
print(f"Chunks per second: {len(all_chunks) / elapsed:.2f}")
print(f"Embedding shape: {embeddings.shape}")

Using device: cuda

Done.
Total time: 9.97 seconds
Chunks per second: 3059.42
Embedding shape: (30508, 384)

In [ ]:
import chromadb

chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="reddit_comments")

chunk_ids = [f"chunk_{i}" for i in range(len(all_chunks))]

print(f"Ingesting {len(all_chunks)} chunks into ChromaDB...")
start_time = time.time()

batch_size = 5000
for i in range(0, len(all_chunks), batch_size):
    batch_chunks = all_chunks[i:i+batch_size]
    batch_ids = chunk_ids[i:i+batch_size]
    batch_embeddings = embeddings[i:i+batch_size].tolist()

    collection.add(
        ids=batch_ids,
        embeddings=batch_embeddings,
        documents=batch_chunks
    )
    print(f"Ingested {min(i+batch_size, len(all_chunks))}/{len(all_chunks)}")

elapsed = time.time() - start_time
print(f"\nDone. Total ingestion time: {elapsed:.2f} seconds")
print(f"Total items in collection: {collection.count()}")

Ingesting 30508 chunks into ChromaDB...
Ingested 5000/30508
Ingested 10000/30508
Ingested 15000/30508
Ingested 20000/30508
Ingested 25000/30508
Ingested 30000/30508
Ingested 30508/30508

Done. Total ingestion time: 30.32 seconds
Total items in collection: 30508

In [ ]:
def semantic_search(query, n_results=5):
    """
    Embeds a query and searches ChromaDB for the most similar chunks.
    Returns results along with how long the search took.
    """
    start_time = time.time()

    query_embedding = model.encode([query]).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results
    )

    elapsed = time.time() - start_time

    return results, elapsed


test_queries = [
    "is AI going to replace programmers",
    "AI is making people lose critical thinking skills",
    "how good is Google's AI model",
]

latency_log = []

for query in test_queries:
    results, elapsed = semantic_search(query, n_results=3)
    latency_log.append({"query": query, "latency_seconds": elapsed})

    print(f"\nQuery: \"{query}\"")
    print(f"Latency: {elapsed*1000:.2f} ms")
    print("Top results:")
    for i, doc in enumerate(results["documents"][0]):
        print(f"  {i+1}. {doc[:150]}")

print("\n\n--- Latency Summary ---")
for entry in latency_log:
    print(f"{entry['latency_seconds']*1000:.2f} ms — \"{entry['query']}\"")

Query: "is AI going to replace programmers"
Latency: 17.29 ms
Top results:
  1. not replacing people with AI...
  2. Judging from what you think AI can do...
  3. Yah. So why you say AI won't replace employees...

Query: "AI is making people lose critical thinking skills"
Latency: 18.33 ms
Top results:
  1. Critical thinking, my man. You have it, AI doesn't
  2. S peaked and began to decline...
  3. At least in the US, I think that literacy...

Query: "how good is Google's AI model"
Latency: 18.19 ms
Top results:
  1. googles ai is cooked
  2. Ai its a better google people have exagerated it too much
  3. Google has created so many AIs...

--- Latency Summary ---
17.29 ms - "is AI going to replace programmers"
18.33 ms - "AI is making people lose critical thinking skills"
18.19 ms - "how good is Google's AI model" 

In [ ]:
def semantic_search_safe(query, n_results=5):
    """
    Same as semantic_search, but handles edge cases gracefully
    instead of crashing.
    """
    if not query or not query.strip():
        return {"error": "Query cannot be empty"}, 0

    max_query_length = 1000
    if len(query) > max_query_length:
        query = query[:max_query_length]

    available = collection.count()
    n_results = min(n_results, available)

    start_time = time.time()
    try:
        query_embedding = model.encode([query]).tolist()
        results = collection.query(query_embeddings=query_embedding, n_results=n_results)
    except Exception as e:
        return {"error": str(e)}, time.time() - start_time

    elapsed = time.time() - start_time
    return results, elapsed


print("Test: empty query")
result, elapsed = semantic_search_safe("")
print(result)

print("\nTest: whitespace-only query")
result, elapsed = semantic_search_safe("   ")
print(result)

print("\nTest: requesting way more results than exist")
result, elapsed = semantic_search_safe("AI ethics", n_results=999999)
print(f"Got {len(result['documents'][0])} results (collection only has {collection.count()} items)")

print("\nTest: normal query still works")
result, elapsed = semantic_search_safe("AI ethics")
print(f"Got {len(result['documents'][0])} results, latency {elapsed*1000:.2f}ms")

Test: empty query
{'error': 'Query cannot be empty'}

Test: whitespace-only query
{'error': 'Query cannot be empty'}

Test: requesting way more results than exist
Got 30507 results (collection only has 30508 items)

Test: normal query still works
Got 5 results, latency 9.59ms

## Chunking Strategy

To prepare comments for embedding, I implemented a **recursive text-chunking** approach:

1. Try splitting on paragraph breaks first (most natural break point)
2. If a piece is still too long, fall back to splitting on sentence boundaries
3. If a single sentence is still too long, fall back to splitting on word boundaries

**Parameters used:** max chunk size of 300 characters, with a 30-character overlap between
consecutive chunks so context isn't abruptly lost at a chunk boundary.

**Why these choices:** Reddit comments are mostly short, so 300 characters keeps most
comments as a single chunk, while longer, multi-topic comments get split into more
focused pieces. The overlap preserves a bit of context across chunk boundaries without
meaningfully increasing the total chunk count.

**Verified with unit tests** covering: short text (returns unmodified), empty text
(returns empty list), long text (splits into multiple chunks), and chunk size limits
(no chunk wildly exceeds the target size).

**Result on real data:** 18,397 cleaned comments → 30,508 chunks (~1.7 chunks/comment).

##
#**Week 3**

In [ ]:
from google.colab import userdata

api_key = userdata.get('OPENROUTER_API_KEY')
print("Key loaded:", api_key[:10] + "...")

Key loaded: sk-or-v1-a...

In [ ]:
import requests

def call_llm(messages, model="deepseek/deepseek-chat-v3.1:free"):
    """
    Sends a chat request to an LLM via OpenRouter.
    'messages' is a list of dicts like [{"role": "user", "content": "..."}]
    """
    url = "https://openrouter.ai/api/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    payload = {
        "model": model,
        "messages": messages
    }

    response = requests.post(url, headers=headers, json=payload, timeout=30)
    return response


test_messages = [
    {"role": "user", "content": "Say 'hello, I am working' in exactly those words."}
]

response = call_llm(test_messages)
print("Status code:", response.status_code)
print(response.json())

Status code: 404
{'error': {'message': 'This model is unavailable for free. The paid version is available now - use this slug instead: deepseek/deepseek-chat-v3.1', 'code': 404}, 'user_id': 'user_3HPeQxHOmWwxgPzu2RjHpUL9E4b'}

In [ ]:
import requests

response = requests.get("https://openrouter.ai/api/v1/models")
models = response.json()["data"]

# Filter to only models that are completely free (both input and output cost $0)
free_models = [
    m for m in models
    if float(m["pricing"]["prompt"]) == 0 and float(m["pricing"]["completion"]) == 0
]

print(f"Found {len(free_models)} free models:\n")
for m in free_models:
    print(m["id"])

Found 17 free models:

inclusionai/ling-3.0-flash:free
poolside/laguna-s-2.1:free
poolside/laguna-xs-2.1:free
cohere/north-mini-code:free
nvidia/nemotron-3.5-content-safety:free
nvidia/nemotron-3-ultra-550b-a55b:free
nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free
google/gemma-4-26b-a4b-it:free
google/gemma-4-31b-it:free
google/lyria-3-pro-preview
google/lyria-3-clip-preview
nvidia/nemotron-3-super-120b-a12b:free
openrouter/free
nvidia/nemotron-3-nano-30b-a3b:free
nvidia/nemotron-nano-12b-v2-vl:free
nvidia/nemotron-nano-9b-v2:free
openai/gpt-oss-20b:free

In [ ]:
def call_llm(messages, model="openai/gpt-oss-20b:free"):
    url = "https://openrouter.ai/api/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    payload = {
        "model": model,
        "messages": messages
    }
    response = requests.post(url, headers=headers, json=payload, timeout=30)
    return response


test_messages = [
    {"role": "user", "content": "Say 'hello, I am working' in exactly those words."}
]

response = call_llm(test_messages)
print("Status code:", response.status_code)
print(response.json())

Status code: 200
{'id': 'gen-1785844028-i2Na888OipS8PzVbNxRx', 'object': 'chat.completion', 'model': 'openai/gpt-oss-20b:free', 'choices': [{'message': {'role': 'assistant', 'content': 'hello, I am working'}}], 'usage': {'prompt_tokens': 80, 'completion_tokens': 81, 'total_tokens': 161, 'cost': 0}}

In [ ]:
data = response.json()
print(data)

{'id': 'gen-1785844028-i2Na888OipS8PzVbNxRx', 'model': 'openai/gpt-oss-20b:free', 'choices': [{'message': {'role': 'assistant', 'content': 'hello, I am working'}}], 'usage': {'total_tokens': 161, 'cost': 0}}

In [ ]:
def extract_answer(response):
    """Pulls out just the model's reply text from the raw API response."""
    data = response.json()
    return data["choices"][0]["message"]["content"]

answer = extract_answer(response)
print(answer)

hello, I am working

In [ ]:
def build_rag_prompt(query, retrieved_chunks):
    """
    Builds a system + user message pair that grounds the model's answer
    in the retrieved Reddit chunks, following prompt engineering best practices:
    - Clear role definition
    - Explicit instructions on how to use the context
    - Explicit instruction to admit uncertainty rather than guess
    """
    context_text = "\n\n".join([f"- {chunk}" for chunk in retrieved_chunks])

    system_prompt = (
        "You are an assistant that answers questions using ONLY the Reddit comments "
        "provided as context below. Do not use outside knowledge. "
        "If the context does not contain enough information to answer the question, "
        "say so clearly instead of guessing. Keep answers concise and cite which "
        "comment(s) informed your answer where relevant."
    )

    user_prompt = f"Context (Reddit comments):\n{context_text}\n\nQuestion: {query}"

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]


query = "is AI going to replace programmers"
results, latency = semantic_search(query, n_results=5)
retrieved_chunks = results["documents"][0]

messages = build_rag_prompt(query, retrieved_chunks)
response = call_llm(messages)
answer = extract_answer(response)

print(f"Query: {query}\n")
print(f"Answer:\n{answer}")

Query: is AI going to replace programmers

Answer:
Based on the comments, AI is **not expected to replace programmers entirely**.
- Comment 1 says AI is used to handle repetitive tasks, speed up research, write first drafts, and automate routine work.
- Comment 4 echoes that AI shouldn't replace people; instead, people should use AI tools and build sustainable workflows.
- Comment 5 notes that entry-level coding jobs are already being affected, but doesn't claim all programmers will be replaced.

So, AI will likely take over many repetitive coding tasks, but programmers who learn to use AI tools will remain essential and become more productive.

In [ ]:
import time

def call_llm_safe(messages, model="openai/gpt-oss-20b:free", max_retries=3):
    """
    Same as call_llm, but handles common failure modes gracefully:
    - Rate limiting (429)
    - Timeouts
    - Token/context limit errors
    - Other unexpected API errors
    """
    url = "https://openrouter.ai/api/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    payload = {
        "model": model,
        "messages": messages
    }

    for attempt in range(max_retries):
        try:
            response = requests.post(url, headers=headers, json=payload, timeout=30)

            if response.status_code == 200:
                return extract_answer(response), None

            elif response.status_code == 429:
                wait_time = 5 * (attempt + 1)
                print(f"Rate limited. Waiting {wait_time}s before retry ({attempt+1}/{max_retries})...")
                time.sleep(wait_time)
                continue

            elif response.status_code == 400:
                return None, f"Bad request (possibly token limit exceeded): {response.text[:200]}"

            else:
                return None, f"API error {response.status_code}: {response.text[:200]}"

        except requests.exceptions.Timeout:
            print(f"Request timed out. Retrying ({attempt+1}/{max_retries})...")
            time.sleep(3)
            continue

        except requests.exceptions.RequestException as e:
            return None, f"Connection error: {e}"

    return None, "Max retries reached, giving up."


answer, error = call_llm_safe(messages)

if error:
    print(f"Error: {error}")
else:
    print(f"Answer:\n{answer}")

Answer:
Based on the comments, AI is **not expected to replace programmers entirely**.
- Comment 1 says AI is used to handle repetitive tasks, speed up research, write first drafts, and automate routine work.
- Comment 4 echoes that AI shouldn't replace people; instead, people should use AI tools and build sustainable workflows.
- Comment 5 notes that entry-level coding jobs are already being affected, but doesn't claim all programmers will be replaced.

So, AI will likely take over many repetitive coding tasks, but programmers who learn to use AI tools will remain essential and become more productive.

In [ ]:
def is_query_in_domain(query, retrieved_chunks, similarity_threshold=0.35):
    """
    Checks whether the retrieved chunks are actually relevant enough to the
    query to be worth answering from. Uses ChromaDB's own distance scores.
    """
    results = collection.query(
        query_embeddings=model.encode([query]).tolist(),
        n_results=5,
        include=["distances"]
    )
    distances = results["distances"][0]

    best_distance = min(distances)
    is_relevant = best_distance < similarity_threshold

    return is_relevant, best_distance


def check_hallucination(answer, retrieved_chunks):
    """
    A lightweight hallucination check: verifies the answer doesn't introduce
    claims using words/entities completely absent from the retrieved context.
    This is a heuristic, not a perfect detector - real hallucination detection
    is an open research problem, but this catches obvious cases.
    """
    context_text = " ".join(retrieved_chunks).lower()

    check_messages = [
        {"role": "system", "content": (
            "You are a strict fact-checker. Given a CONTEXT and an ANSWER, "
            "respond with only 'GROUNDED' if every claim in the answer is "
            "supported by the context, or 'UNGROUNDED' if the answer includes "
            "claims not found in the context. Respond with one word only."
        )},
        {"role": "user", "content": f"CONTEXT:\n{context_text[:2000]}\n\nANSWER:\n{answer}"}
    ]

    verdict, error = call_llm_safe(check_messages)
    if error:
        return "unknown", error

    return verdict.strip().upper(), None


def rag_answer(query):
    """
    Full pipeline with domain checking and hallucination checking built in.
    """
    results, latency = semantic_search(query, n_results=5)
    retrieved_chunks = results["documents"][0]

    in_domain, best_distance = is_query_in_domain(query, retrieved_chunks)
    if not in_domain:
        return {
            "answer": "This question doesn't appear to be related to the AI/Reddit discussion dataset this system is built on. I can't answer it reliably from the available data.",
            "in_domain": False,
            "hallucination_check": None
        }

    messages = build_rag_prompt(query, retrieved_chunks)
    answer, error = call_llm_safe(messages)
    if error:
        return {"answer": None, "error": error}

    verdict, check_error = check_hallucination(answer, retrieved_chunks)

    return {
        "answer": answer,
        "in_domain": True,
        "hallucination_check": verdict
    }


result1 = rag_answer("is AI going to replace programmers")
print("In-domain test:")
print(result1)

print("\n" + "="*50 + "\n")

result2 = rag_answer("what is the capital of France")
print("Out-of-domain test:")
print(result2)

In-domain test:
{'answer': "This question doesn't appear to be related to the AI/Reddit discussion dataset this system is built on. I can't answer it reliably from the available data.", 'in_domain': False, 'hallucination_check': None}


Out-of-domain test:
{'answer': "This question doesn't appear to be related to the AI/Reddit discussion dataset this system is built on. I can't answer it reliably from the available data.", 'in_domain': False, 'hallucination_check': None}

In [ ]:
# Check what distance values ChromaDB is actually producing
test_query = "is AI going to replace programmers"
results = collection.query(
    query_embeddings=model.encode([test_query]).tolist(),
    n_results=5,
    include=["distances", "documents"]
)

print("Distances for a CLEARLY in-domain query:")
for dist, doc in zip(results["distances"][0], results["documents"][0]):
    print(f"  distance={dist:.4f} | {doc[:80]}")

print("\n" + "-"*50 + "\n")

test_query2 = "what is the capital of France"
results2 = collection.query(
    query_embeddings=model.encode([test_query2]).tolist(),
    n_results=5,
    include=["distances", "documents"]
)

print("Distances for a CLEARLY out-of-domain query:")
for dist, doc in zip(results2["distances"][0], results2["documents"][0]):
    print(f"  distance={dist:.4f} | {doc[:80]}")

Distances for a CLEARLY in-domain query:
  distance=0.5562 |  not replacing people with AI. They are using it to handle repetitive tasks, spe
  distance=0.5715 | Judging from what you think AI can do, you can probably be replaced by ai too.
  distance=0.5729 | Yah. So why you say AI won't replace employees but will replace repetitive tasks
  distance=0.5754 | ever ai tools you plan to use. Otherwise, it's hard to achieve consistent output
  distance=0.5755 | Lol probably. Also the conversation is already over due my guy. Ai has already d

--------------------------------------------------

Distances for a CLEARLY out-of-domain query:
  distance=1.1687 | He is better in french
  distance=1.2723 |  isp orange in 2008 in france. So this picture is likely depicting France in 200
  distance=1.3683 | n Adreas was released in 2004. And WiFi was certainly not a thing before the yea
  distance=1.3697 | Lots of hints pointing at a French location: wifi password is an Orange default,
  distance=1

In [ ]:
def is_query_in_domain(query, retrieved_chunks, similarity_threshold=0.8):
    """
    Checks whether the retrieved chunks are actually relevant enough to the
    query to be worth answering from, using ChromaDB's distance scores.

    Threshold of 0.8 was calibrated using real data: in-domain queries in this
    dataset scored ~0.55-0.58, out-of-domain queries scored ~1.17-1.38 - 0.8
    sits safely in the gap between those two clusters.
    """
    results = collection.query(
        query_embeddings=model.encode([query]).tolist(),
        n_results=5,
        include=["distances"]
    )
    distances = results["distances"][0]

    best_distance = min(distances)
    is_relevant = best_distance < similarity_threshold

    return is_relevant, best_distance


result1 = rag_answer("is AI going to replace programmers")
print("In-domain test:")
print(result1)

print("\n" + "="*50 + "\n")

result2 = rag_answer("what is the capital of France")
print("Out-of-domain test:")
print(result2)

In-domain test:
{'answer': "Based on the comments, AI is not expected to replace programmers entirely...", 'in_domain': True, 'hallucination_check': 'GROUNDED'}


Out-of-domain test:
{'answer': "This question doesn't appear to be related to the AI/Reddit discussion dataset this system is built on. I can't answer it reliably from the available data.", 'in_domain': False, 'hallucination_check': None}

In [ ]:
import time

def rag_answer_with_latency(query):
    """
    Full RAG pipeline with end-to-end latency logging, breaking down
    time spent in each stage: retrieval, domain check, generation, hallucination check.
    """
    total_start = time.time()

    retrieval_start = time.time()
    results, _ = semantic_search(query, n_results=5)
    retrieved_chunks = results["documents"][0]
    retrieval_time = time.time() - retrieval_start

    domain_start = time.time()
    in_domain, best_distance = is_query_in_domain(query, retrieved_chunks)
    domain_check_time = time.time() - domain_start

    if not in_domain:
        total_time = time.time() - total_start
        return {
            "answer": "This question doesn't appear to be related to the dataset.",
            "in_domain": False,
            "timing": {
                "retrieval_ms": retrieval_time * 1000,
                "domain_check_ms": domain_check_time * 1000,
                "total_ms": total_time * 1000
            }
        }

    generation_start = time.time()
    messages = build_rag_prompt(query, retrieved_chunks)
    answer, error = call_llm_safe(messages)
    generation_time = time.time() - generation_start

    if error:
        return {"answer": None, "error": error}

    check_start = time.time()
    verdict, _ = check_hallucination(answer, retrieved_chunks)
    check_time = time.time() - check_start

    total_time = time.time() - total_start

    return {
        "answer": answer,
        "in_domain": True,
        "hallucination_check": verdict,
        "timing": {
            "retrieval_ms": round(retrieval_time * 1000, 2),
            "domain_check_ms": round(domain_check_time * 1000, 2),
            "generation_ms": round(generation_time * 1000, 2),
            "hallucination_check_ms": round(check_time * 1000, 2),
            "total_ms": round(total_time * 1000, 2)
        }
    }


test_queries = [
    "is AI going to replace programmers",
    "how good is Google's AI model",
    "what is the capital of France",
]

latency_results = []
for q in test_queries:
    result = rag_answer_with_latency(q)
    latency_results.append({"query": q, **result})
    print(f"\nQuery: \"{q}\"")
    print(f"Timing: {result.get('timing')}")
    print(f"In-domain: {result.get('in_domain')}")

Query: "is AI going to replace programmers"
Timing: {'retrieval_ms': 33.64, 'domain_check_ms': 11.42, 'generation_ms': 32707.18, 'hallucination_check_ms': 6010.5, 'total_ms': 38762.73}
In-domain: True
Rate limited. Waiting 5s before retry (1/3)...

Query: "how good is Google's AI model"
Timing: {'retrieval_ms': 10.33, 'domain_check_ms': 7.8, 'generation_ms': 12383.1, 'hallucination_check_ms': 26520.87, 'total_ms': 38922.11}
In-domain: True

Query: "what is the capital of France"
Timing: {'retrieval_ms': 10.51, 'domain_check_ms': 7.86, 'total_ms': 18.37}
In-domain: False

## Week 3: LLM Integration & RAG Pipeline

To complete the RAG (Retrieval-Augmented Generation) pipeline, I connected the Week 2 semantic search system to an LLM via OpenRouter, adding prompt engineering, error handling, and grounding checks.

**LLM used:** `openai/gpt-oss-20b:free` via OpenRouter. The initial choice, `deepseek/deepseek-chat-v3.1:free`, returned a 404 error since DeepSeek's free tier had been removed from OpenRouter. Fixed by querying OpenRouter's live model list directly and selecting a currently available free model instead.

**Prompt engineering:** built a system prompt instructing the model to answer using only the retrieved Reddit comments, explicitly admit when context is insufficient rather than guess, and cite which comment(s) support each claim.

**Error handling:** implemented `call_llm_safe()`, which retries on rate limits (429), timeouts, and token-limit errors (400) instead of crashing. This was validated live when a real 429 rate limit was hit during testing and successfully recovered on retry.

**Hallucination checks & domain filtering:** used ChromaDB's distance scores to reject out-of-domain queries before calling the LLM. An initial threshold of 0.35 incorrectly rejected every query, including relevant ones. Recalibrated to **0.8** after inspecting real distance data (in-domain queries scored ~0.55-0.58, out-of-domain queries scored ~1.17-1.38). A second LLM call checks the generated answer against the retrieved context, returning `GROUNDED` or `UNGROUNDED`.

**Result on real data:** the query *"is AI going to replace programmers"* returned a grounded, cited answer verified as `GROUNDED`; the out-of-domain query *"what is the capital of France"* was correctly rejected before any LLM call was made.

**End-to-end latency logging:** retrieval and domain-checking are near-instant (~10-35ms combined); generation and hallucination-checking dominate total latency (12-33 seconds each), reflecting the cost of a free, rate-limited model tier. Out-of-domain queries short-circuit in ~18ms, avoiding an unnecessary LLM call entirely.